In [8]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *
from utils import process_greek

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Greece Shapefiles"

In [9]:
import warnings
warnings.filterwarnings("ignore")

In [10]:
NUTS0 = 'GR'
NUTS2 = 'CMacedonia'

In [11]:
YEAR = 2023
MONTH = 'August'
PERIOD = '1st'

In [12]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))

In [13]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,area,population,population_density,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,22.46197,40.60446,2023-08-01,κεντρικης μακεδονιας,αλεξανδρειας,1,8,31,2023,0.201299,0.97953,-0.866025,-0.5,-0.508531,-0.861044,478.8,38293.0,86.8,46.40326,0.611207,0.249284,-0.532555,-0.249284,0.599636,0.238649,-0.526441,-0.238649,0.081898,0.079507,0.055983,0.079507,26.605909,31.743636,21.468182,11.510466,1.708510,11.778876,0.439805,21.186067,5.575849,24.421964,8.937268,0.000000,0.000000,187.946484,18483.883785,1629.598106,4,120.472193,7.866005,180.191380,0.0,1.400071,31,99,36,99.0,30,99,12,12,1,6,7,2,0,74,0,0
1,22.07722,41.01522,2023-08-01,κεντρικης μακεδονιας,αλμωπιας,1,8,31,2023,0.201299,0.97953,-0.866025,-0.5,-0.508531,-0.861044,985.8,24924.0,28.0,46.57952,0.617810,0.191093,-0.540819,-0.191093,0.619570,0.195880,-0.540852,-0.195880,0.051798,0.048885,0.034330,0.048885,23.988889,29.154074,18.823704,9.238694,1.920162,9.573419,-0.371554,15.120878,3.834306,16.704092,6.474975,0.000000,0.000000,315.981766,25701.766035,44.089931,2,194.572797,144.613300,181.767317,0.0,113.550432,31,93,36,94.0,30,93,12,12,3,5,8,2,0,95,0,0
2,22.89198,40.65603,2023-08-01,κεντρικης μακεδονιας,αμπελοκηπων μενεμενης,1,8,31,2023,0.201299,0.97953,-0.866025,-0.5,-0.508531,-0.861044,9.8,49674.0,5319.1,46.65786,0.113069,-0.034805,-0.143001,0.034805,0.154827,-0.024789,-0.182593,0.024789,0.063708,0.027689,0.055943,0.027689,31.700000,38.170000,25.230000,12.381429,3.596667,11.870000,2.525385,18.760769,5.121429,23.898000,11.870000,0.000000,0.000000,416.613286,1013.723139,1259.926671,2,166.963790,4.655965,180.457193,0.0,24.343007,32,97,9,99.0,30,97,13,13,10,8,9,2,0,138,0,0
3,23.95900,40.91196,2023-08-01,κεντρικης μακεδονιας,αμφιπολης,1,8,31,2023,0.201299,0.97953,-0.866025,-0.5,-0.508531,-0.861044,411.8,7169.0,22.3,47.41120,0.408284,0.095824,-0.396856,-0.095824,0.358557,0.085568,-0.347080,-0.085568,0.065723,0.054425,0.052317,0.054425,27.240667,33.196667,21.284667,10.655161,3.113550,9.895287,0.416503,16.682835,4.861432,20.539981,7.233860,0.000000,0.000000,127.404946,14413.246350,196.335052,5,220.810138,250.710581,185.823044,0.0,25.464784,31,88,30,88.0,30,88,10,10,1,6,6,2,0,124,0,0
4,23.69747,40.49593,2023-08-01,κεντρικης μακεδονιας,αριστοτελη,1,8,31,2023,0.201299,0.97953,-0.866025,-0.5,-0.508531,-0.861044,747.0,16994.0,24.5,46.92004,0.660292,0.286775,-0.565762,-0.286775,0.646614,0.276531,-0.555794,-0.276531,0.055642,0.054578,0.039862,0.054578,24.437222,28.469444,20.405000,10.968774,3.770478,9.453781,2.661346,15.362634,4.844974,17.259672,7.381085,1.943263,1.943263,341.156538,11468.297401,1749.385768,22,273.587292,768.043224,181.562024,0.0,1.033661,14,99,10,99.0,10,99,4,4,6,4,4,2,0,73,0,0


In [14]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

scaler = MinMaxScaler()
imputer = KNNImputer()

X_test = scaler.fit_transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)

In [15]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,22.90326,40.67609,κορδελιου ευοσμου,1,8,2023,0.992797
1,22.91743,40.42897,θερμαϊκου,1,8,2023,0.990260
2,22.72632,40.61725,δελτα,1,8,2023,0.989307
3,22.95369,40.62334,θεσσαλονικης,1,8,2023,0.982542
4,22.89198,40.65603,αμπελοκηπων μενεμενης,1,8,2023,0.978483
5,23.27201,41.13057,ηρακλειας,1,8,2023,0.971732
6,22.95724,40.57876,καλαμαριας,1,8,2023,0.971136
7,22.95341,40.68195,παυλου μελα,1,8,2023,0.968910
8,23.19403,40.32484,νεας προποντιδας,1,8,2023,0.952338
9,23.08410,40.49006,θερμης,1,8,2023,0.945237


In [16]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.0648252687608289, 0.3787123377107355, 0.6803072708219687, 0.838221147727126, 0.935474790938512, 1.0]


In [17]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,22.90326,40.67609,κορδελιου ευοσμου,1,8,2023,0.992797,5
1,22.91743,40.42897,θερμαϊκου,1,8,2023,0.990260,5
2,22.72632,40.61725,δελτα,1,8,2023,0.989307,5
3,22.95369,40.62334,θεσσαλονικης,1,8,2023,0.982542,5
4,22.89198,40.65603,αμπελοκηπων μενεμενης,1,8,2023,0.978483,5
5,23.27201,41.13057,ηρακλειας,1,8,2023,0.971732,5
6,22.95724,40.57876,καλαμαριας,1,8,2023,0.971136,5
7,22.95341,40.68195,παυλου μελα,1,8,2023,0.968910,5
8,23.19403,40.32484,νεας προποντιδας,1,8,2023,0.952338,5
9,23.08410,40.49006,θερμης,1,8,2023,0.945237,5


In [18]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}_(new).csv", encoding = enc, index = False)

In [19]:
##TODO Visualisation of results